# A.R.I.A. — Adaptive Road Intelligence Architecture
## 3-Stage Curriculum Training · YOLOv11n · Severity Detection

| Stage | Dataset | Purpose |
|---|---|---|
| 1 | RDD2022 all countries | Global foundation — learn all 3 severity classes |
| 2 | RDD2022 India only | Regional specialisation |
| 3 | IIT Madras | Indian city road fine-tune |

**Output classes:** `0 damage_low` · `1 damage_medium` · `2 damage_high`

Each stage warm-starts from the previous stage's `best.pt`.


## Cell 1 — Install

In [1]:
!pip install -q ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 25.3 MB/s eta 0:00:00


## Cell 2 — Imports & Path Discovery

In [2]:
# %% [Cell 2] Imports, path discovery, directory tree
import os
import json
import shutil
from pathlib import Path
from collections import Counter
from concurrent.futures import ThreadPoolExecutor

import torch
from ultralytics import YOLO
from ultralytics import __version__ as ul_version

print(f"Ultralytics : {ul_version}")
print(f"PyTorch     : {torch.__version__}")
print(f"CUDA        : {torch.cuda.is_available()} — "
      f"{torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")

CONFIG_PATH = Path("/kaggle/working/aria_config.json")

def find_rdd_split(root="/kaggle/input") -> Path | None:
    for dp, dns, _ in os.walk(root):
        dl = {d.lower(): d for d in dns}
        if "rdd_split" in dl:
            return Path(dp) / dl["rdd_split"]
    return None

def find_itt_madras(root="/kaggle/input") -> Path | None:
    for dp, dns, fns in os.walk(root):
        if "data.yaml" not in fns:
            continue
        cand = Path(dp)
        img_dir = cand / "train" / "images"
        if img_dir.exists() and any(img_dir.iterdir()):
            return cand
    return None

RDD_SPLIT = find_rdd_split()
ITT_ROOT  = find_itt_madras()

print("\n" + "=" * 64)
print("/kaggle/input/  (2 levels)")
print("=" * 64)
for dp, dns, fns in os.walk("/kaggle/input"):
    depth = dp.replace("/kaggle/input", "").count(os.sep)
    if depth > 2:
        dns.clear()
        continue
    indent = "  " * depth
    print(f"{indent}{os.path.basename(dp) or 'input'}/  [{len(fns)} files]")

print(f"\nRDD_SPLIT  : {RDD_SPLIT}")
print(f"ITT Madras : {ITT_ROOT}")

if RDD_SPLIT is None or ITT_ROOT is None:
    raise RuntimeError(
        "One or both datasets not found. Check the tree above and update "
        "find_rdd_split / find_itt_madras if folder names differ."
    )

print("\n" + "=" * 64)
print("RDD2022 split counts")
print("=" * 64)
for sp in ("train", "val", "test"):
    img_d = RDD_SPLIT / sp / "images"
    lbl_d = RDD_SPLIT / sp / "labels"
    imgs = len(list(img_d.glob("*"))) if img_d.exists() else 0
    lbls = len(list(lbl_d.glob("*.txt"))) if lbl_d.exists() else 0
    print(f"  {sp:<6}: {imgs:>6} images   {lbls:>6} labels")

print("\nCountry prefixes in RDD2022/train/images/:")
pc = Counter()
for p in (RDD_SPLIT / "train" / "images").glob("*"):
    parts = p.stem.split("_")
    prefix = "_".join(x for x in parts if not x.isdigit())
    pc[prefix] += 1
for prefix, cnt in sorted(pc.items(), key=lambda x: -x[1]):
    print(f"  {prefix:<30}: {cnt:>5} images")

print(f"\nITT Madras split counts ({ITT_ROOT.name}):")
for sp in ("train", "valid", "test"):
    d = ITT_ROOT / sp / "images"
    cnt = len(list(d.glob("*"))) if d.exists() else 0
    print(f"  {sp:<6}: {cnt:>5} images")

cfg = json.loads(CONFIG_PATH.read_text()) if CONFIG_PATH.exists() else {}
cfg.update({"rdd_split": str(RDD_SPLIT), "itt_root": str(ITT_ROOT)})
CONFIG_PATH.write_text(json.dumps(cfg, indent=2))
print(f"\n✓ Paths saved to {CONFIG_PATH}")


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics : 8.4.21
PyTorch     : 2.9.0+cu126
CUDA        : True — Tesla P100-PCIE-16GB

/kaggle/input/  (2 levels)
input/  [0 files]
  datasets/  [0 files]
    dptel22/  [0 files]
    aliabdelmenam/  [0 files]

RDD_SPLIT  : /kaggle/input/datasets/aliabdelmenam/rdd-2022/RDD_SPLIT
ITT Madras : /kaggle/input/datasets/dptel22/itt-madras

RDD2022 split counts
  train :  26869 images    26869 labels
  val   :   5758 images     5758 labels
  test  :   5758 images     5758 labels

Country prefixes in RDD2022/train/images/:
  Japan                         :  7432 images
  Norway                        :  5708 images
  India                         :  5368 images
  United_States          

## Cell 3 — Stage 1 Data Prep: ALL RDD2022 → 3 Severity Classes

In [3]:
# %% [Cell 3] Stage 1 data preparation — ALL RDD2022, 3 severity classes
#
# FIX (crash):   import os added — os.symlink needs it even if Cell 2 ran
# FIX (disk):    os.symlink instead of shutil.copy2 for images — zero disk cost
# FIX (race):    symlink wrapped in try/except FileExistsError
# FIX (labels):  empty .txt written for background images (no label file)
# FIX (unknown): unknown RDD class IDs are counted and reported, not silently remapped
# CLEANUP:       shutil removed from imports (no longer used here)
import os
import json
from pathlib import Path
from collections import Counter
from concurrent.futures import ThreadPoolExecutor

CONFIG_PATH = Path("/kaggle/working/aria_config.json")
cfg         = json.loads(CONFIG_PATH.read_text())
RDD_SPLIT   = Path(cfg["rdd_split"])               # /kaggle/input/  READ-ONLY
STAGE1_DIR  = Path("/kaggle/working/stage1_data")  # /kaggle/working/ WRITE HERE

SPLIT_MAP = {
    RDD_SPLIT / "train": STAGE1_DIR / "train",
    RDD_SPLIT / "val":   STAGE1_DIR / "valid",
}
for dst in SPLIT_MAP.values():
    (dst / "images").mkdir(parents=True, exist_ok=True)
    (dst / "labels").mkdir(parents=True, exist_ok=True)

# RDD2022 original class → severity tier
# 0 D00 longitudinal crack → 0 damage_low
# 1 D10 transverse crack   → 0 damage_low
# 2 D20 alligator crack    → 1 damage_medium
# 3 D40 pothole            → 2 damage_high
# 4 D44 pothole/manhole    → 2 damage_high
RDD_TO_SEVERITY = {0: 0, 1: 0, 2: 1, 3: 2, 4: 2}

def _scan_class_ids(lbl_dir: Path) -> set:
    ids = set()
    if not lbl_dir.exists():
        return ids
    for f in lbl_dir.glob("*.txt"):
        try:
            for line in f.read_text().strip().splitlines():
                p = line.split()
                if p:
                    ids.add(int(p[0]))
        except Exception:
            pass
    return ids

all_ids: set = set()
for src in SPLIT_MAP:
    all_ids |= _scan_class_ids(src / "labels")
print(f"Original RDD2022 class IDs : {sorted(all_ids)}")
unknown_ids = all_ids - set(RDD_TO_SEVERITY.keys())
if unknown_ids:
    print(f"[WARN] Unknown class IDs (will map to damage_low): {sorted(unknown_ids)}")
print(f"Remapping → 0:damage_low  1:damage_medium  2:damage_high\n")

def _symlink_and_remap(task):
    img_src, dst_img, src_lbl, dst_lbl = task
    unknown_count = 0
    try:
        # IMAGES: symlink from /kaggle/input/ → zero disk usage
        try:
            os.symlink(img_src, dst_img)
        except FileExistsError:
            pass  # already linked — safe to ignore

        # LABELS: physically write remapped .txt to /kaggle/working/
        if src_lbl.exists():
            lines = []
            for line in src_lbl.read_text().strip().splitlines():
                p = line.strip().split()
                if len(p) >= 5:
                    orig_cls = int(p[0])
                    if orig_cls not in RDD_TO_SEVERITY:
                        unknown_count += 1
                    new_cls = RDD_TO_SEVERITY.get(orig_cls, 0)
                    lines.append(f"{new_cls} " + " ".join(p[1:]))
            dst_lbl.write_text("\n".join(lines))
        else:
            # Background image — write empty .txt so YOLO counts it as negative sample
            dst_lbl.write_text("")

        return True, None, unknown_count
    except Exception as e:
        return False, f"{img_src.name}: {e}", 0

total_ok = total_err = total_bg = total_unknown = 0
for src_split, dst_split in SPLIT_MAP.items():
    src_img_dir = src_split / "images"
    src_lbl_dir = src_split / "labels"
    if not src_img_dir.exists():
        raise FileNotFoundError(
            f"Images directory not found: {src_img_dir}\n"
            "Re-run Cell 2 to verify paths."
        )
    tasks = [
        (img,
         dst_split / "images" / img.name,
         src_lbl_dir / (img.stem + ".txt"),           # READ from /kaggle/input/
         dst_split / "labels" / (img.stem + ".txt"))  # WRITE to /kaggle/working/
        for img in src_img_dir.glob("*") if img.is_file()
    ]
    ok = err = unk = 0
    with ThreadPoolExecutor(max_workers=8) as pool:
        for success, msg, u in pool.map(_symlink_and_remap, tasks):
            if success:
                ok += 1
            else:
                err += 1
                print(f"  [ERROR] {msg}")
            unk += u

    bg = sum(1 for f in (dst_split / "labels").glob("*.txt") if f.read_text().strip() == "")
    print(f"  {dst_split.name:<8}: {ok:>6} linked  {bg:>5} background  {unk:>4} unknown-cls  {err} errors")
    total_ok += ok; total_err += err; total_bg += bg; total_unknown += unk

if total_unknown > 0:
    print(f"\n[WARN] {total_unknown} labels had unknown RDD class IDs — mapped to damage_low(0)")

(STAGE1_DIR / "data.yaml").write_text(
    f"path: {STAGE1_DIR}\ntrain: train/images\nval: valid/images\n"
    "nc: 3\nnames: ['damage_low', 'damage_medium', 'damage_high']\n"
)

train_n = len(list((STAGE1_DIR / "train" / "images").glob("*")))
valid_n = len(list((STAGE1_DIR / "valid" / "images").glob("*")))
print(f"\nStage 1 dataset ready:")
print(f"  train      : {train_n:>6} images")
print(f"  valid      : {valid_n:>6} images")
print(f"  background : {total_bg:>6} no-damage samples")
print(f"  errors     : {total_err}")

if train_n == 0:
    raise RuntimeError(f"0 images in stage1 train. Source: {RDD_SPLIT / 'train' / 'images'}")

cfg["stage1"] = {"train": train_n, "valid": valid_n, "background": total_bg}
CONFIG_PATH.write_text(json.dumps(cfg, indent=2))
print(f"✓ Stage 1 data.yaml → {STAGE1_DIR / 'data.yaml'}")


Original RDD2022 class IDs : [0, 1, 2, 3, 4]
Remapping → 0:damage_low  1:damage_medium  2:damage_high

  train   :  26869 linked   8097 background     0 unknown-cls  0 errors
  valid   :   5758 linked   1837 background     0 unknown-cls  0 errors

Stage 1 dataset ready:
  train      :  26869 images
  valid      :   5758 images
  background :   9934 no-damage samples
  errors     : 0
✓ Stage 1 data.yaml → /kaggle/working/stage1_data/data.yaml


## Cell 4 — Stage 1 Training: `yolo11n.pt` → `aria_stage1.pt`

In [4]:
# %% [Cell 4] Stage 1 training — global road damage foundation
# FIX: optimizer='SGD' explicit (auto silently ignored lr0)
# FIX: epochs=50 (model hadn't converged at 40)
# FIX: cache=False (disk too full for disk cache)
# FIX: patience=10 (early stopping — default 100 wasted GPU time)
# FIX: batch=-1 (auto-batch — fills P100 VRAM, faster than fixed batch=16)
# CLEANUP: stage1_data deleted after weights saved to free disk for Stage 2
import json, shutil
from pathlib import Path
from ultralytics import YOLO

CONFIG_PATH    = Path("/kaggle/working/aria_config.json")
STAGE1_DIR     = Path("/kaggle/working/stage1_data")
STAGE1_WEIGHTS = Path("/kaggle/working/aria_stage1.pt")
cfg = json.loads(CONFIG_PATH.read_text())

model   = YOLO("yolo11n.pt")
results = model.train(
    data          = str(STAGE1_DIR / "data.yaml"),
    epochs        = 50,
    imgsz         = 640,
    batch         = -1,       # auto-batch: fills GPU VRAM optimally
    device        = 0,
    workers       = 4,
    name          = "aria_stage1",
    project       = "/kaggle/working/runs",
    exist_ok      = True,
    # ── optimizer ─────────────────────────────────────────────
    optimizer     = "SGD",
    lr0           = 0.01,
    momentum      = 0.937,
    weight_decay  = 0.0005,
    # ── schedule ──────────────────────────────────────────────
    cos_lr        = True,
    warmup_epochs = 3,
    patience      = 10,       # early stopping — stop if no improvement for 10 epochs
    # ── mosaic ────────────────────────────────────────────────
    close_mosaic  = 10,
    # ── augmentation ──────────────────────────────────────────
    fliplr        = 0.5,
    degrees       = 5.0,
    translate     = 0.1,
    scale         = 0.2,
    hsv_h         = 0.015,
    hsv_s         = 0.4,
    hsv_v         = 0.4,
    mixup         = 0.1,
    copy_paste    = 0.1,
    erasing       = 0.2,
    # ── memory ────────────────────────────────────────────────
    amp           = True,
    cache         = False,    # disk full — no caching
    plots         = True,
)

best_pt = Path(results.save_dir) / "weights" / "best.pt"
shutil.copy2(best_pt, STAGE1_WEIGHTS)

stage1_map50 = float(results.results_dict.get("metrics/mAP50(B)", 0.0))
cfg["stage1_mAP50"] = stage1_map50
CONFIG_PATH.write_text(json.dumps(cfg, indent=2))

# CLEANUP: remove stage1_data to free disk space for Stage 2
print("\nCleaning up stage1_data to free disk...")
shutil.rmtree(STAGE1_DIR, ignore_errors=True)
print("  ✓ stage1_data removed")

print(f"\n{'─'*48}")
print(f"  Stage 1  mAP@50 : {stage1_map50:.4f}")
print(f"  Saved   → {STAGE1_WEIGHTS}")
print(f"{'─'*48}")


Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/stage1_data/data.yaml, degrees=5.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.2, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.4, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=aria_stage1, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patien

## Cell 5 — Stage 2 Data Prep: IIT Madras → 3 Severity Classes

In [5]:
# %% [Cell 7] Stage 3 data preparation — IIT Madras, 3 severity classes
#
# FIX (crash):   import os added
# FIX (disk):    os.symlink instead of shutil.copy2 for images
# FIX (race):    symlink wrapped in try/except FileExistsError
# FIX (dynamic): reads data.yaml, matches by name not index, handles 1/3/other nc
# FIX (fuzzy):   prints side-by-side expected vs actual class names for easy verification
# FIX (labels):  empty .txt for background images
# CLEANUP:       shutil removed from imports
import os
import json
import yaml
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

CONFIG_PATH = Path("/kaggle/working/aria_config.json")
cfg         = json.loads(CONFIG_PATH.read_text())
ITT_ROOT    = Path(cfg["itt_root"])                # /kaggle/input/  READ-ONLY
STAGE3_DIR  = Path("/kaggle/working/stage3_data")  # /kaggle/working/ WRITE HERE

for sp in ("train", "valid"):
    (STAGE3_DIR / sp / "images").mkdir(parents=True, exist_ok=True)
    (STAGE3_DIR / sp / "labels").mkdir(parents=True, exist_ok=True)

# ── FIX: Read and validate IIT Madras classes BEFORE touching any data ─────────
try:
    itt_yaml  = yaml.safe_load((ITT_ROOT / "data.yaml").read_text())
    itt_names = itt_yaml.get("names", [])
    itt_nc    = itt_yaml.get("nc", len(itt_names))
except Exception as e:
    raise RuntimeError(f"Could not read IIT Madras data.yaml: {e}")

print(f"IIT Madras data.yaml — nc={itt_nc}")
print(f"Classes found: {itt_names}")
print()

if itt_nc == 1:
    print(f"[WARN] Only 1 class ('{itt_names[0]}') — mapping all to damage_high (2)")
    ITT_TO_SEVERITY = {0: 2}

elif itt_nc == 3:
    name_to_idx = {n.lower().strip(): i for i, n in enumerate(itt_names)}

    # Expected names and their severity tier
    severity_rules = {
        "longitudinal crack": 0,   # damage_low
        "crocodile crack":    1,   # damage_medium
        "pothole":            2,   # damage_high
    }

    ITT_TO_SEVERITY = {}
    unmatched = []
    for expected_name, sev in severity_rules.items():
        if expected_name in name_to_idx:
            ITT_TO_SEVERITY[name_to_idx[expected_name]] = sev
        else:
            unmatched.append(expected_name)

    # FIX: print side-by-side for instant visual verification
    print("Class mapping verification:")
    print(f"  {'Actual name in data.yaml':<28} {'Expected match':<28} {'→ Severity'}")
    print(f"  {'─'*28} {'─'*28} {'─'*12}")
    for i, actual in enumerate(itt_names):
        mapped_sev = ITT_TO_SEVERITY.get(i, "⚠ UNMATCHED")
        sev_label  = ["damage_low","damage_medium","damage_high"][mapped_sev] if isinstance(mapped_sev, int) else mapped_sev
        match_key  = next((k for k, v in severity_rules.items() if name_to_idx.get(k) == i), "— no match —")
        print(f"  {actual:<28} {match_key:<28} → {sev_label}")

    if unmatched:
        print(f"\n[WARN] Unmatched expected names: {unmatched}")
        print("Filling unmatched IIT classes with damage_low (0).")
        for i in range(itt_nc):
            if i not in ITT_TO_SEVERITY:
                ITT_TO_SEVERITY[i] = 0

else:
    print(f"[WARN] Unexpected nc={itt_nc}. Mapping all classes to damage_high (2).")
    ITT_TO_SEVERITY = {i: 2 for i in range(itt_nc)}

print(f"\nFinal ITT_TO_SEVERITY: {ITT_TO_SEVERITY}")

# ── Resolve split folder names (handles 'valid' vs 'val') ─────────────────────
def _resolve_split(root: Path, name: str) -> Path | None:
    for candidate in (name, "val" if name == "valid" else "valid"):
        if (root / candidate / "images").exists():
            return root / candidate
    return None

SPLIT_MAP = {}
for sp in ("train", "valid"):
    src = _resolve_split(ITT_ROOT, sp)
    if src:
        SPLIT_MAP[sp] = (src, STAGE3_DIR / sp)
    else:
        print(f"[WARN] ITT Madras '{sp}' split not found — skipping")

if not SPLIT_MAP:
    raise RuntimeError("No valid splits found in IIT Madras dataset.")

# ── Copy (symlink) images, write remapped labels ───────────────────────────────
def _symlink_and_remap(task):
    img_src, dst_img, src_lbl, dst_lbl = task
    try:
        try:
            os.symlink(img_src, dst_img)
        except FileExistsError:
            pass
        if src_lbl.exists():
            lines = []
            for line in src_lbl.read_text().strip().splitlines():
                p = line.strip().split()
                if len(p) >= 5:
                    new_cls = ITT_TO_SEVERITY.get(int(p[0]), 0)
                    lines.append(f"{new_cls} " + " ".join(p[1:]))
            dst_lbl.write_text("\n".join(lines))
        else:
            dst_lbl.write_text("")
        return True, None
    except Exception as e:
        return False, str(e)

total_err = total_bg = 0
for sp_name, (src_split, dst_split) in SPLIT_MAP.items():
    tasks = [
        (img,
         dst_split / "images" / img.name,
         (src_split / "labels") / (img.stem + ".txt"),
         dst_split / "labels" / (img.stem + ".txt"))
        for img in (src_split / "images").glob("*") if img.is_file()
    ]
    ok = err = 0
    with ThreadPoolExecutor(max_workers=8) as pool:
        for success, msg in pool.map(_symlink_and_remap, tasks):
            ok += success
            if not success:
                err += 1
                print(f"  [ERROR] {msg}")

    bg = sum(1 for f in (dst_split / "labels").glob("*.txt") if f.read_text().strip() == "")
    print(f"  {sp_name:<6}: {ok:>5} linked  {bg:>4} background  {err} errors")
    total_err += err
    total_bg  += bg

(STAGE3_DIR / "data.yaml").write_text(
    f"path: {STAGE3_DIR}\ntrain: train/images\nval: valid/images\n"
    "nc: 3\nnames: ['damage_low', 'damage_medium', 'damage_high']\n"
)

train_n = len(list((STAGE3_DIR / "train" / "images").glob("*")))
valid_n = len(list((STAGE3_DIR / "valid" / "images").glob("*")))
print(f"\nStage 3 dataset ready:")
print(f"  train      : {train_n:>5} images")
print(f"  valid      : {valid_n:>5} images")
print(f"  background : {total_bg:>5} no-damage samples")
print(f"  errors     : {total_err}")

if train_n == 0:
    raise RuntimeError("0 images in stage3 train — check ITT Madras path.")

cfg["stage3"] = {"train": train_n, "valid": valid_n, "background": total_bg}
CONFIG_PATH.write_text(json.dumps(cfg, indent=2))
print(f"✓ Stage 3 data.yaml → {STAGE3_DIR / 'data.yaml'}")


IIT Madras data.yaml — nc=3
Classes found: ['crocodile crack', 'longitudinal crack', 'pothole']

Class mapping verification:
  Actual name in data.yaml     Expected match               → Severity
  ──────────────────────────── ──────────────────────────── ────────────
  crocodile crack              crocodile crack              → damage_medium
  longitudinal crack           longitudinal crack           → damage_low
  pothole                      pothole                      → damage_high

Final ITT_TO_SEVERITY: {1: 0, 0: 1, 2: 2}
  train :  1906 linked     1 background  0 errors
  valid :   542 linked     0 background  0 errors

Stage 3 dataset ready:
  train      :  1906 images
  valid      :   542 images
  background :     1 no-damage samples
  errors     : 0
✓ Stage 3 data.yaml → /kaggle/working/stage3_data/data.yaml


## Cell 6 — Stage 2 Training: `aria_stage2.pt` → `aria_best_v1.pt`

In [6]:
# %% [Cell 6] Stage 3 training — IIT Madras fine-tune → aria_best_v1.pt
# CHANGE: loads aria_stage1.pt directly (Stage 2 removed — nano model too small for 3-stage)
# CHANGE: freeze=10 (was 16 — too aggressive, backbone needs to adapt to IIT Madras domain)
# FIX: optimizer='AdamW' (better for low-lr fine-tuning on small dataset)
# FIX: patience=10 (early stopping)
# FIX: batch=-1 (auto-batch)
import json, shutil
from pathlib import Path
from ultralytics import YOLO

CONFIG_PATH    = Path("/kaggle/working/aria_config.json")
STAGE3_DIR     = Path("/kaggle/working/stage3_data")
STAGE1_WEIGHTS = Path("/kaggle/working/aria_stage1.pt")   # ← was stage2
FINAL_WEIGHTS  = Path("/kaggle/working/aria_best_v1.pt")
cfg = json.loads(CONFIG_PATH.read_text())

if not STAGE1_WEIGHTS.exists():
    raise FileNotFoundError(f"Stage 1 weights not found: {STAGE1_WEIGHTS}. Run Cell 4 first.")

model   = YOLO(str(STAGE1_WEIGHTS))                       # ← was stage2
results = model.train(
    data          = str(STAGE3_DIR / "data.yaml"),
    epochs        = 30,
    imgsz         = 640,
    batch         = -1,
    device        = 0,
    workers       = 4,
    name          = "aria_stage3",
    project       = "/kaggle/working/runs",
    exist_ok      = True,
    freeze        = 10,        # ← was 16 — let upper backbone adapt to IIT Madras domain
    # ── optimizer — AdamW better for low-lr fine-tuning ───────
    optimizer     = "AdamW",
    lr0           = 0.001,
    weight_decay  = 0.0005,
    # ── schedule ──────────────────────────────────────────────
    cos_lr        = True,
    close_mosaic  = 5,
    patience      = 10,
    # ── augmentation (conservative for small dataset) ─────────
    fliplr        = 0.5,
    degrees       = 5.0,
    translate     = 0.1,
    scale         = 0.2,
    hsv_h         = 0.015,
    hsv_s         = 0.4,
    hsv_v         = 0.4,
    mixup         = 0.0,
    copy_paste    = 0.0,
    erasing       = 0.2,
    # ── memory ────────────────────────────────────────────────
    amp           = True,
    cache         = True,
    plots         = True,
)

best_pt = Path(results.save_dir) / "weights" / "best.pt"
shutil.copy2(best_pt, FINAL_WEIGHTS)

stage3_map50 = float(results.results_dict.get("metrics/mAP50(B)", 0.0))
cfg["stage3_mAP50"] = stage3_map50
CONFIG_PATH.write_text(json.dumps(cfg, indent=2))

print(f"\n{'─'*48}")
print(f"  Stage 2  mAP@50 : {stage3_map50:.4f}")
print(f"  Saved   → {FINAL_WEIGHTS}")
print(f"{'─'*48}")


Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=5, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/stage3_data/data.yaml, degrees=5.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.2, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, half=False, hsv_h=0.015, hsv_s=0.4, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/kaggle/working/aria_stage1.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=aria_stage3, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overla

## Cell 7 — Final Evaluation & Results

In [7]:
# %% [Cell 7] Final validation — IIT Madras val + RDD2022 test (held-out)
# CHANGE: added RDD2022 test split evaluation for true generalization metric
# CHANGE: removed Stage 2 row from summary (Stage 2 no longer exists)
import os
import json
from pathlib import Path
from ultralytics import YOLO

CONFIG_PATH   = Path("/kaggle/working/aria_config.json")
FINAL_WEIGHTS = Path("/kaggle/working/aria_best_v1.pt")
cfg = json.loads(CONFIG_PATH.read_text())
RDD_SPLIT = Path(cfg["rdd_split"])

if not FINAL_WEIGHTS.exists():
    raise FileNotFoundError(f"Final weights not found: {FINAL_WEIGHTS}. Run Cell 6 first.")

model = YOLO(str(FINAL_WEIGHTS))

# ── 1. Validate on IIT Madras val set (same as training val) ──────────────────
print("=" * 66)
print("  Validating on IIT Madras val set (training val set)")
print("=" * 66)
metrics_itt = model.val(
    data    = "/kaggle/working/stage3_data/data.yaml",
    device  = 0,
    imgsz   = 640,
    plots   = True,
)
itt_map50   = float(metrics_itt.box.map50)
itt_map5095 = float(metrics_itt.box.map)

# ── 2. Validate on RDD2022 test split (truly held-out) ────────────────────────
rdd_test_img = RDD_SPLIT / "test" / "images"
rdd_test_lbl = RDD_SPLIT / "test" / "labels"

rdd_test_map50   = float("nan")
rdd_test_map5095 = float("nan")

if rdd_test_img.exists() and any(rdd_test_img.iterdir()):
    # Build a temporary data.yaml pointing to RDD2022 test split
    # Need to symlink + remap labels to our 3-class severity scheme
    TEST_DIR = Path("/kaggle/working/rdd_test_eval")
    (TEST_DIR / "images").mkdir(parents=True, exist_ok=True)
    (TEST_DIR / "labels").mkdir(parents=True, exist_ok=True)

    RDD_TO_SEVERITY = {0: 0, 1: 0, 2: 1, 3: 2, 4: 2}

    test_count = 0
    for img in rdd_test_img.glob("*"):
        if not img.is_file():
            continue
        dst_img = TEST_DIR / "images" / img.name
        dst_lbl = TEST_DIR / "labels" / (img.stem + ".txt")
        # Symlink image
        try:
            os.symlink(img, dst_img)
        except FileExistsError:
            pass
        # Remap label
        src_lbl = rdd_test_lbl / (img.stem + ".txt")
        if src_lbl.exists():
            lines = []
            for line in src_lbl.read_text().strip().splitlines():
                p = line.strip().split()
                if len(p) >= 5:
                    new_cls = RDD_TO_SEVERITY.get(int(p[0]), 0)
                    lines.append(f"{new_cls} " + " ".join(p[1:]))
            dst_lbl.write_text("\n".join(lines))
        else:
            dst_lbl.write_text("")
        test_count += 1

    # Write test data.yaml — use test split as BOTH train and val
    # (we're only calling model.val(), YOLO reads the 'val' key)
    (TEST_DIR / "data.yaml").write_text(
        f"path: {TEST_DIR}\n"
        f"train: images\n"      # required by YOLO but unused for val()
        f"val: images\n"
        "nc: 3\n"
        "names: ['damage_low', 'damage_medium', 'damage_high']\n"
    )

    print(f"\n{'=' * 66}")
    print(f"  Validating on RDD2022 TEST split ({test_count} images — truly held-out)")
    print(f"{'=' * 66}")

    metrics_rdd = model.val(
        data    = str(TEST_DIR / "data.yaml"),
        device  = 0,
        imgsz   = 640,
        plots   = True,
    )
    rdd_test_map50   = float(metrics_rdd.box.map50)
    rdd_test_map5095 = float(metrics_rdd.box.map)

    # Cleanup test eval data
    import shutil
    shutil.rmtree(TEST_DIR, ignore_errors=True)
else:
    print(f"\n[WARN] RDD2022 test split not found at {rdd_test_img} — skipping test eval")

# ── Save all metrics ──────────────────────────────────────────────────────────
s1 = cfg.get("stage1_mAP50", float("nan"))
s3 = cfg.get("stage3_mAP50", float("nan"))

cfg["final_itt_mAP50"]     = itt_map50
cfg["final_itt_mAP5095"]   = itt_map5095
cfg["final_rdd_test_mAP50"]   = rdd_test_map50
cfg["final_rdd_test_mAP5095"] = rdd_test_map5095
CONFIG_PATH.write_text(json.dumps(cfg, indent=2))

# ── Print final report ────────────────────────────────────────────────────────
W = 70
print(f"\n{'═'*W}")
print(f"  A.R.I.A.  Road Damage Detection — v2")
print(f"  Architecture : YOLOv11n (2-stage curriculum)")
print(f"  Classes      : damage_low · damage_medium · damage_high")
print(f"  Model        : {FINAL_WEIGHTS.name}")
print(f"{'─'*W}")
print(f"  IIT Madras val set (training val — optimistic)")
print(f"    mAP@50    : {itt_map50:.4f}")
print(f"    mAP@50-95 : {itt_map5095:.4f}")
print(f"{'─'*W}")
if not (rdd_test_map50 != rdd_test_map50):  # not NaN
    print(f"  RDD2022 test split (held-out — TRUE generalization)")
    print(f"    mAP@50    : {rdd_test_map50:.4f}")
    print(f"    mAP@50-95 : {rdd_test_map5095:.4f}")
    print(f"{'─'*W}")
print(f"  Training progression")
print(f"{'─'*W}")
print(f"  {'Stage':<48} {'Val set':<14} {'mAP@50':>8}")
print(f"  {'─'*48} {'─'*14} {'─'*8}")
print(f"  {'1 — RDD2022 all countries (global foundation)':<48} {'RDD2022 val':<14} {s1:>8.4f}")
print(f"  {'2 — IIT Madras fine-tune  (final model)':<48} {'IIT Madras':<14} {s3:>8.4f}")
print(f"{'═'*W}")
print(f"\n✅ Final model saved: {FINAL_WEIGHTS}")
print(   "   Download aria_best_v1.pt from the Output tab.")


  Validating on IIT Madras val set (training val set)
Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
YOLO11n summary (fused): 101 layers, 2,582,737 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 117.6±74.2 MB/s, size: 88.1 KB)
val: Scanning /kaggle/working/stage3_data/valid/labels.cache... 542 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 542/542 151.6Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 34/34 6.6it/s 5.1s
                   all        542       1853      0.385      0.331      0.303      0.126
            damage_low        107        284      0.297      0.271      0.213     0.0727
         damage_medium         63        125      0.354      0.153      0.149     0.0471
           damage_high        533       1444      0.505      0.569      0.547      0.258
Speed: 1.3ms preprocess, 3.1ms inference, 0.0ms loss, 1.4ms 